# Simple RAG pipeline (Google Colab)

Same flow as `rag_pipeline.py`: load docs → split → embed with OpenAI → retrieve → answer.

**Colab differences vs the local script:**
1. Install packages in the notebook (`%pip install`).
2. Read `OPENAI_API_KEY` from Colab Secrets (or prompt for it). No `.env` file.
3. Create the sample docs in the notebook so you do not need to upload the `docs/` folder.

**Before you run:** in Colab, open the key icon (Secrets) on the left, add `OPENAI_API_KEY`, and enable notebook access.

In [ ]:
%pip install -q langchain-openai langchain-core langchain-community langchain-text-splitters

In [ ]:
import os
from getpass import getpass

try:
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Loaded OPENAI_API_KEY from Colab Secrets.")
except Exception:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
    print("Loaded OPENAI_API_KEY from prompt.")

In [ ]:
from pathlib import Path

docs_dir = Path("docs")
docs_dir.mkdir(exist_ok=True)

(docs_dir / "crashloopbackoff.txt").write_text(
    "Kubernetes CrashLoopBackOff\n\n"
    "A pod is stuck restarting because its container keeps crashing.\n"
    "Check: kubectl describe pod, kubectl logs, and recent events.\n",
    encoding="utf-8",
)
(docs_dir / "imagepullbackoff.txt").write_text(
    "Kubernetes ImagePullBackOff\n\n"
    "The node cannot pull the container image.\n"
    "Common causes include a wrong image name, a missing tag, or registry auth.\n"
    "Check: kubectl describe pod and the image pull events.\n",
    encoding="utf-8",
)

print("Wrote sample files:", list(docs_dir.glob("*.txt")))

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Load documents
loader = DirectoryLoader(
    "docs",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)
docs = loader.load()
print(f"Loaded {len(docs)} document(s)")

# 2. Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=40)
chunks = splitter.split_documents(docs)
print(f"Split into {len(chunks)} chunk(s)")

# 3. Embed with OpenAI and store in memory
embeddings = OpenAIEmbeddings()
vectorstore = InMemoryVectorStore.from_documents(chunks, embeddings)

# 4. Retrieve the most relevant chunks for a question
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})


def format_docs(retrieved_docs):
    return "\n\n".join(doc.page_content for doc in retrieved_docs)


# 5. Prompt the model to answer only from retrieved context
prompt = ChatPromptTemplate.from_template(
    "Answer the question using only the context below. "
    "If the context does not contain the answer, say you do not know.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}"
)

llm = ChatOpenAI(model="gpt-4.1-mini")
parser = StrOutputParser()

# 6. Compose the RAG chain: retrieve → prompt → model → string
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | parser
)

In [ ]:
# 7. Ask a question
question = "Why is my pod in CrashLoopBackOff and how do I debug it?"
print(f"Question: {question}\n")

retrieved = retriever.invoke(question)
print("Retrieved chunks:")
for i, chunk in enumerate(retrieved):
    print(f"--- chunk {i} ({chunk.metadata.get('source', 'unknown')}) ---")
    print(chunk.page_content)

print("\nAnswer:")
print(rag_chain.invoke(question))

### Optional: use your own files instead

Skip the sample-docs cell and upload `.txt` files into `docs/`:

```python
from google.colab import files
from pathlib import Path

Path("docs").mkdir(exist_ok=True)
uploaded = files.upload()  # pick .txt files in the dialog
for name, data in uploaded.items():
    Path("docs", name).write_bytes(data)
```